##### Imports

In [ ]:
import os, sys, json
import asyncio
from langchain_anthropic import ChatAnthropic
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.messages import HumanMessage, ToolMessage

##### Start the MCP Server

In [ ]:
# ===========================================================
# CELL 1 — Start the server first!
#
# In a separate terminal run:
#   python complete_mcp_server.py
# ===========================================================

mcp = MultiServerMCPClient({
    "school": {
        "url": "http://127.0.0.1:8000/mcp",
        "transport": "streamable_http",
    }
})
SERVER_URL = "http://127.0.0.1:8000/mcp"
print("Setup complete!")


###### Select the LLM Model

In [ ]:
llm = ChatAnthropic(
    model='claude-sonnet-4-5-20250929',
    anthropic_api_key="")

##### Listing All Tools

In [ ]:
# ===========================================================
# CELL 2 — LIST ALL TOOLS (with JSON-RPC explanation)
#
# get_tools() sends this JSON-RPC message to the server:
# {
#   "jsonrpc": "2.0",
#   "method": "tools/list",
#   "params": {}
# }
# ===========================================================

tools = await mcp.get_tools()

print("TOOLS — Functions Claude can CALL")
print(f"\nTotal tools found: {len(tools)}\n")

for i, tool in enumerate(tools, 1):
    print(f"  Tool {i}: {tool.name}")
    print(f"  Description: {tool.description}")
    print(f"  Input Schema: {tool.args if hasattr(tool, 'args_schema') else 'see args'}")
    print()

print("""
 What happened under the hood (JSON-RPC):

  YOUR NOTEBOOK sent:
  {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list",
    "params": {}
  }

  SERVER replied with all tool names, descriptions, and schemas.
  LangChain converted them into Python tool objects for you.
""")

##### Listing All Resources

In [ ]:
# ===========================================================
# CELL 3 — LIST ALL RESOURCES
#
# Resources are data the server exposes for Claude to READ.
# They are like files or database records.
#
# JSON-RPC method used: "resources/list"
# ===========================================================

# Access the raw MCP session to list resources directly
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def list_resources():
    async with streamablehttp_client("http://127.0.0.1:8000/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # This sends: {"jsonrpc":"2.0","method":"resources/list","params":{}}
            resources = await session.list_resources()
            return resources

result = await list_resources()

print("   RESOURCES — Data Claude can READ")
print(f"\nTotal resources found: {len(result.resources)}\n")

for i, resource in enumerate(result.resources, 1):
    print(f"  Resource {i}: {resource.name}")
    print(f"  URI:         {resource.uri}")
    print(f"  Description: {resource.description}")
    print()

print("""
 What happened under the hood (JSON-RPC):

  YOUR NOTEBOOK sent:
  {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "resources/list",
    "params": {}
  }

  SERVER replied with all resource URIs and descriptions.
""")


##### Reading the Resource

In [ ]:
# ===========================================================
# CELL 4 — READ A SPECIFIC RESOURCE
#
# After listing resources, you can READ one by its URI.
# JSON-RPC method: "resources/read"
# ===========================================================

async def read_resource(uri: str):
    async with streamablehttp_client("http://127.0.0.1:8000/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # This sends: {"jsonrpc":"2.0","method":"resources/read","params":{"uri":"..."}}
            content = await session.read_resource(uri)
            return content

print("READING RESOURCES")

# Read the school info resource
school_info = await read_resource("resource://school-info")
print("resource://school-info")
print("-" * 40)
for item in school_info.contents:
    print(item.text)

# Read the class schedule resource
schedule = await read_resource("resource://class-schedule")
print("resource://class-schedule")
print("-" * 40)
for item in schedule.contents:
    print(item.text)

# Read a specific student resource (dynamic URI with parameter)
student = await read_resource("resource://student/ali")
print("resource://student/ali")
print("-" * 40)
for item in student.contents:
    print(item.text)




In [ ]:
# # ===========================================================
# # CELL 4.1 — OPERATION : SUBSCRIBE to a resource
# #
# # Get notified when a resource's content changes
# # Useful for live data — prices, scores, status updates
# # JSON-RPC method: resources/subscribe
# # ===========================================================

# async def subscribe_to_resource(uri: str):
#     async with streamablehttp_client(SERVER_URL) as (read, write, _):
#         async with ClientSession(read, write) as session:
#             await session.initialize()
            
#             # Subscribe to get notified when this resource changes
#             await session.subscribe_resource(uri)
#             print(f"Subscribed to: {uri}")
#             print("   Server will notify us when this resource changes")
#             print("   Useful for: live scores, prices, status updates")

# await subscribe_to_resource("resource://school-info")

# print("""
# JSON-RPC sent:
# {
#   "jsonrpc": "2.0",
#   "method": "resources/subscribe",
#   "params": { "uri": "resource://school-info" }
# }

# When the resource changes, server sends:
# {
#   "jsonrpc": "2.0",
#   "method": "notifications/resources/updated",
#   "params": { "uri": "resource://school-info" }
# }
# """)

##### Listing all Prompts

In [ ]:
# ===========================================================
# CELL 5 — LIST ALL PROMPTS
#
# Prompts are pre-written templates the server exposes.
# JSON-RPC method: "prompts/list"
# ===========================================================

async def list_prompts():
    async with streamablehttp_client("http://127.0.0.1:8000/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # This sends: {"jsonrpc":"2.0","method":"prompts/list","params":{}}
            prompts = await session.list_prompts()
            return prompts

prompts_result = await list_prompts()

print("PROMPTS — Templates Claude can USE")
print(f"Total prompts found: {len(prompts_result.prompts)}\n")

for i, prompt in enumerate(prompts_result.prompts, 1):
    print(f"  Prompt {i}: {prompt.name}")
    print(f"  Description: {prompt.description}")
    if prompt.arguments:
        args = [f"{a.name} ({'required' if a.required else 'optional'})"
                for a in prompt.arguments]
        print(f"  Arguments: {', '.join(args)}")
    print()

print("""
 What happened under the hood (JSON-RPC):

  YOUR NOTEBOOK sent:
  {
    "jsonrpc": "2.0",
    "id": 3,
    "method": "prompts/list",
    "params": {}
  }

  SERVER replied with all prompt names, descriptions, and arguments.
""")

##### Using a Prompt

In [ ]:
# ===========================================================
# CELL 6 — GET AND USE A PROMPT
#
# After listing, you can GET a prompt (fill in its variables)
# JSON-RPC method: "prompts/get"
# ===========================================================

async def get_prompt(name: str, arguments: dict):
    async with streamablehttp_client("http://127.0.0.1:8000/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Sends: {"jsonrpc":"2.0","method":"prompts/get",
            #         "params":{"name":"...","arguments":{...}}}
            prompt = await session.get_prompt(name, arguments)
            return prompt

print("USING A PROMPT — explain_concept")

# Get the prompt template filled with our argument
prompt_result = await get_prompt("explain_concept", {"concept": "MCP"})

# Extract the filled prompt text
prompt_text = prompt_result.messages[0].content.text
print(f"\nFilled prompt:\n{prompt_text}")

# Now send this prompt to Claude
print("\n" + "=" * 55)
print("Claude's Response:")
print("=" * 55)

response = llm.invoke([HumanMessage(prompt_text)])
print(response.content)



##### Using a tool

In [ ]:
# ===========================================================
# CELL 7 — USE A TOOL WITH CLAUDE (the normal flow)
# ===========================================================

async def run_agent(question):
    print(f"Qestion: {question}")
    llm_bound = llm.bind_tools(tools)
    messages = [HumanMessage(question)]

    ai_msg = llm_bound.invoke(messages)
    messages.append(ai_msg)

    if not ai_msg.tool_calls:
        print(f"AI: {ai_msg.content}")
        return

    for tc in ai_msg.tool_calls:
        print(f"Tool: {tc['name']} | Args: {tc['args']}")
        tool_obj = next(t for t in tools if t.name == tc["name"])
        result = await tool_obj.ainvoke(tc["args"])
        print(f"Result: {result}")
        messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    final = llm_bound.invoke(messages)
    print(f"💬 {final.content}")

await run_agent("What is the grade of student Ali?")
await run_agent("What is 25 multiplied by 6?")



#### Complete Picture

In [ ]:
# ===========================================================
# CELL 8 — THE COMPLETE JSON-RPC PICTURE
# ===========================================================

print("""
╔══════════════════════════════════════════════════════╗
║         MCP JSON-RPC 2.0 — COMPLETE PICTURE          ║
╠══════════════════════════════════════════════════════╣
║                                                      ║
║  METHOD              WHAT IT DOES                    ║
║  ──────────────────  ──────────────────────────────  ║
║  tools/list          List all available tools        ║
║  tools/call          Call a specific tool            ║
║                                                      ║
║  resources/list      List all available resources    ║
║  resources/read      Read a specific resource        ║
║                                                      ║
║  prompts/list        List all available prompts      ║
║  prompts/get         Get a filled prompt template    ║
║                                                      ║
╠══════════════════════════════════════════════════════╣
║                                                      ║
║  Every message looks like:                           ║
║  {                                                   ║
║    "jsonrpc": "2.0",      ← always this version     ║
║    "id": 1,               ← matches request+reply   ║
║    "method": "tools/list",← what you want to do     ║
║    "params": {}           ← arguments if needed     ║
║  }                                                   ║
║                                                      ║
║  LangChain handles ALL of this automatically.        ║
║  You just call get_tools() and ainvoke().            ║
║                                                      ║
╚══════════════════════════════════════════════════════╝
""")